**knowledge graph agent:

Traditional GraphRAG systems often rely on manual schema design, handcrafted rules, and fixed retrieval strategies, making them brittle and difficult to generalize across domains.

** So till this point we have fed the data split it into triples and then added them as nodes and established a relation which was entirely done manually. from the triples the knowledge graph was formed nd the relationship is assigned at the edges.
** But what if we want everything to be covered by the ai agent following flowchart will enlighten us about the same

User Question
      ↓
AI Agent
      ↓
Knowledge Graph
      ↓
Reasoning
      ↓
Decision
     

**So here an user will qs then the  ai agent will search the appropriate knowledge graph for us and then the reasoning will be done and the decision will be taken and accordingly the actions will be performed.
 ↓
Action


For example user asks the question:"Find me weather information for Kolkata"
Agent thinks as:
1. User needs weather data
2. Search KG
3. Find WeatherAPI
4. Find endpoint
5. Find required parameter
6. Call API
7. Return 
and the task will be performed

so here instead of us writing these code like 
api = discover_api(intent)
endpoint = discover_endpoint(api)
parameter = discover_parameter(endpoin

the ai agent will be able to decide which knowledge graph to choose from the qs and accordingly the task will be established

#to make it  short:
An Agentic Knowledge Graph is an AI system where Large Language Models (LLMs) act autonomously to build, maintain, and traverse a Knowledge Graph. While traditional graphs require manual schema creation and data entry, agentic KGs use a pipeline of specialized AI agents to autonomously extract data, define schemas, and execute multi-hop queriest)answer

In [51]:
import networkx as nx

G = nx.DiGraph()

In [52]:
triples = [

("WeatherAPI","belongs_to","Weather"),
("WeatherAPI","has_endpoint","CurrentWeather"),
("CurrentWeather","requires","city"),
("CurrentWeather","returns","temperature"),

("MapsAPI","belongs_to","Maps"),
("MapsAPI","has_endpoint","Location"),
("Location","requires","place"),
("Location","returns","coordinates"),

("NewsAPI","belongs_to","News"),
("NewsAPI","has_endpoint","Headlines"),
("Headlines","returns","news")

]

In [53]:
for s,r,o in triples:#graph formation with the designed root nodes relations and 

    G.add_edge(
        s,
        o,
        relation=r
    )

In [54]:
def discover_api(intent):

    for source,target,data in G.edges(data=True):

        if (
            data["relation"]=="belongs_to"
            and
            target.lower()==intent.lower()
        ):

            return source

    return None

In [55]:
def discover_endpoint(api):

    for source,target,data in G.edges(data=True):

        if (
            source==api
            and
            data["relation"]=="has_endpoint"
        ):

            return target

    return None

In [56]:
def discover_parameter(endpoint):

    for source,target,data in G.edges(data=True):

        if (
            source==endpoint
            and
            data["relation"]=="requires"
        ):

            return target

    return None

In [57]:
#Agent Planning
#Agents first create a plan
def create_plan(question):

    return [

        "Extract Intent",

        "Find API",

        "Find Endpoint",

        "Find Parameter",

        "Generate Answer"

    ]

In [58]:
plan = create_plan(
    "What is the weather in Kolkata?"
)

for step in plan:
    print(step)

Extract Intent
Find API
Find Endpoint
Find Parameter
Generate Answer


In [59]:
memory = []

In [60]:
def extract_intent(question):

    q = question.lower()

    if "weather" in q:
        return "Weather"

    if "map" in q:
        return "Maps"

    if "news" in q:
        return "News"

    return None

In [61]:
def extract_city(question):

    words = question.split()

    return words[-1].replace("?","")

#Earlier KG Discovery was like:
Question
   ↓
Extract Intent
   ↓
Search KG
   ↓
Find API
   ↓
Stop

#Agentic KG:

Question
   ↓
Extract Intent
   ↓
Find API
   ↓
Find Endpoint
   ↓
Find Parameter
   ↓
Extract Parameter Value(till this i could achieve as the execution plan can only be used once we have the llm use)
   ↓
Store Result/Memory
   ↓
Return Execution Plan


In [49]:
def run_agent(question):

    print("Question:",question)

    intent = extract_intent(question)
    print("Intent Found:",intent)

    api = discover_api(intent)
    print("API Found:",api)

    endpoint = discover_endpoint(api)
    print("Endpoint Found:",endpoint)

    parameter = discover_parameter(endpoint)
    print("Parameter Needed:",parameter)

    value = extract_city(question)
    print("Parameter Value:",value)

    result = {
        "intent": intent,
        "api": api,
        "endpoint": endpoint,
        "parameter": parameter,
        "value": value
    }

    return result

In [62]:
weather_db = {#for testing purpose we can import sqlite3 for real database extraction
    "Kolkata": 34,
    "Delhi": 39,
    "Mumbai": 31,
    "Bangalore": 28
}

In [63]:
def execute_tool(agent_state):

    city = agent_state["value"]

    temperature = weather_db.get(city)

    return {
        "city": city,
        "temperature": temperature
    }

In [64]:
agent_state = run_agent(
    "What is the weather in Kolkata?"#will return all the step by step break down used to process the kg
)

tool_result = execute_tool(#for temperature printing
    agent_state
)

print(tool_result)

Question: What is the weather in Kolkata?
Intent Found: Weather
API Found: WeatherAPI
Endpoint Found: CurrentWeather
Parameter Needed: city
Parameter Value: Kolkata
{'city': 'Kolkata', 'temperature': 34}
